# Final Classification: Baseline vs MiniBatch DL vs K-SVD
* Protocol: leave-one-repetition-out (rep 6 = test), restimulus labels.
* No data leakage: dictionary and scaler fit on train only.

In [1]:
%load_ext autoreload
%autoreload 2
 
import sys
sys.path.append('../') 

import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [2]:
load_dir = '../data/preprocessed/DB5/'
y_windows = np.load(os.path.join(load_dir, 'S01_y_windows.npy'))

# Os rótulos de treino e teste são idênticos para os três métodos graças ao random_state=42
_, _, y_train, y_test = train_test_split(
    np.zeros((len(y_windows), 1)), y_windows, test_size=0.2, random_state=42, stratify=y_windows
)

## 1. BASELINE (CLASSICAL FEATURES)

In [3]:
print("Treinando SVM 1/3: Baseline Clássico...")
X_features_classical = np.load(os.path.join(load_dir, 'S01_X_features_classical.npy'))
X_train_class, X_test_class, _, _ = train_test_split(
    X_features_classical, y_windows, test_size=0.2, random_state=42, stratify=y_windows
)

scaler_class = StandardScaler()
X_train_class_scaled = scaler_class.fit_transform(X_train_class)
X_test_class_scaled = scaler_class.transform(X_test_class)

svm_baseline = SVC(kernel='rbf', C=1.0, random_state=42)
svm_baseline.fit(X_train_class_scaled, y_train)
y_pred_class = svm_baseline.predict(X_test_class_scaled)

Treinando SVM 1/3: Baseline Clássico...


## 2. MINIBATCH DICTIONARY LEARNING

In [4]:
print("Treinando SVM 2/3: MiniBatch DL...")
X_train_mb = np.load(os.path.join(load_dir, 'S01_X_train_sparse.npy'))
X_test_mb = np.load(os.path.join(load_dir, 'S01_X_test_sparse.npy'))

scaler_mb = StandardScaler()
X_train_mb_scaled = scaler_mb.fit_transform(X_train_mb)
X_test_mb_scaled = scaler_mb.transform(X_test_mb)

svm_mb = SVC(kernel='rbf', C=1.0, random_state=42)
svm_mb.fit(X_train_mb_scaled, y_train)
y_pred_mb = svm_mb.predict(X_test_mb_scaled)

Treinando SVM 2/3: MiniBatch DL...


## 3. K-SVD

In [5]:
print("Treinando SVM 3/3: K-SVD Clássico...")
X_train_ksvd = np.load(os.path.join(load_dir, 'S01_X_train_sparse_ksvd.npy'))
X_test_ksvd = np.load(os.path.join(load_dir, 'S01_X_test_sparse_ksvd.npy'))

scaler_ksvd = StandardScaler()
X_train_ksvd_scaled = scaler_ksvd.fit_transform(X_train_ksvd)
X_test_ksvd_scaled = scaler_ksvd.transform(X_test_ksvd)

svm_ksvd = SVC(kernel='rbf', C=1.0, random_state=42)
svm_ksvd.fit(X_train_ksvd_scaled, y_train)
y_pred_ksvd = svm_ksvd.predict(X_test_ksvd_scaled)

Treinando SVM 3/3: K-SVD Clássico...


## FINAL RESULTS

In [6]:
print("\n" + "="*60)
print(" DUELO DE ARQUITETURAS: RESULTADOS FINAIS")
print("="*60)

print(f"Acurácia - Baseline (Features TD)   : {accuracy_score(y_test, y_pred_class) * 100:.2f}%")
print(f"Acurácia - MiniBatch DL             : {accuracy_score(y_test, y_pred_mb) * 100:.2f}%")
print(f"Acurácia - K-SVD Clássico           : {accuracy_score(y_test, y_pred_ksvd) * 100:.2f}%")
print("\n" + "-"*60)


 DUELO DE ARQUITETURAS: RESULTADOS FINAIS
Acurácia - Baseline (Features TD)   : 84.31%
Acurácia - MiniBatch DL             : 60.43%
Acurácia - K-SVD Clássico           : 60.60%

------------------------------------------------------------
